#Transform Results Data
1. Read bronze results table
2. Keep only the columns required for analytics (Drop url column)
3. Standardize column names using snake_case (driverId → driver_id, constructorId → constructor_id,raceName → race_name,positiontext → finish_position_text )
4.Rename columns to make them more meaningful (date → race_date, grid → grid_position, laps → completed_laps, number → car_number, position → finish_position 
5. Filter out rows where season, round, constructor_id and driver_id is null (business key validation)
6. Remove duplicate records
7. Transform column values of race_name to Title Case
8. Write the transformed data to silver results table

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment_config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.results"
silver_table = f"{catalog_name}.{silver_schema}.results"


Step 1,2,3,4-Read bronze results table,Select only required columns and standardize the column names

In [0]:
from pyspark.sql import functions as F
results_df = (
  spark.table(bronze_table)
       .filter((F.col("batch_id") == v_batch_id))
       .select("season",
                "round",
                "constructorId",
                "driverId",
                "date",
                "raceName",
                "grid",
                "laps",
                "number",
                "points",
                "position",
                "positionText",
                "status",
                "ingestion_timestamp",
                "source_file",
                "batch_id")
       .withColumnsRenamed({
            "constructorId": "constructor_id",
            "driverId": "driver_id",
            "raceName": "race_name",
            "date": "race_date",
            "grid": "grid_position",
            "laps": "completed_laps",
            "number": "car_number",
            "position": "final_position",
            "positionText": "final_position_text"
        })
)

In [0]:
results_df.display()

Step 5,6- Apply Data Quality checks
- Filter out rows where season, round, constructor_id and driver_id is null (business key validation)
- Remove duplicate records

In [0]:
results_valid_df = (
    results_df
        .filter(
            F.col("season").isNotNull() &
            F.col("round").isNotNull() &
            F.col("constructor_id").isNotNull() &
            F.col("driver_id").isNotNull() 
        )
        .dropDuplicates(["season", "round", "constructor_id", "driver_id"])
)

In [0]:
display(results_df.count() - results_valid_df.count())

In [0]:
results_valid_df.display()

Step 7- Transform column values of race_name to Title Case

In [0]:
results_final_df = (results_valid_df
                            .withColumn("race_name",F.initcap(F.col("race_name")))

)
results_final_df.display()



Step 8- Write the transformed data to silver results table

In [0]:
write_to_silver(
    input_df=results_final_df,
    target_table=silver_table,
    merge_condition="t.season = s.season AND t.round = s.round AND t.constructor_id = s.constructor_id AND t.driver_id = s.driver_id",
    columns_to_update=[
        "race_name",
        "race_date",
        "grid_position",
        "completed_laps",
        "car_number",
        "points",
        "final_position",
        "final_position_text",
        "status",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)

In [0]:
display(spark.table(silver_table))